<!-- # Elliptic Curves: From Real Numbers to Finite Fields

## 1. The Curve Equation and Singularity Check

An elliptic curve over the real numbers ($\mathbb{R}$) is defined by the Weierstrass equation:
$$y^2 = x^3 + ax + b$$

For the curve to form a valid, non-singular group (meaning it has no cusps or self-intersections), its discriminant must not be zero:
$$\Delta = -4a^3 - 27b^2 \neq 0$$

When transitioning to a **Finite Field ($\mathbb{F}_p$)**, where $p$ is a prime number, the curve is no longer a continuous, smooth line but a set of discrete points on a grid. The equation becomes a congruence modulo $p$:
$$y^2 \equiv x^3 + ax + b \pmod p$$

Similarly, the singularity check must be evaluated under modulo $p$ to ensure no singular points exist in the finite field:
$$-4a^3 - 27b^2 \not\equiv 0 \pmod p$$

## 2. Arithmetic in Finite Fields: The Modular Inverse

In real numbers, calculating the geometric slope $m$ involves division. However, standard division does not exist in modular arithmetic. Instead of dividing by a number $k$, we multiply by its **modular multiplicative inverse**, denoted as $k^{-1}$.

### Fermat's Little Theorem
To find the inverse $k^{-1}$, we use **Fermat's Little Theorem**, which states that for any prime $p$ and integer $k$ not divisible by $p$:
$$k^{p-1} \equiv 1 \pmod p$$

By multiplying both sides by $k^{-1}$, we can algebraically isolate the modular inverse:
$$k^{-1} \equiv k^{p-2} \pmod p$$
Thus, "dividing" by $k$ modulo $p$ is mathematically equivalent to multiplying by $k^{p-2} \pmod p$.

### Binary Exponentiation
Calculating $k^{p-2}$ directly for cryptographically large primes is computationally infeasible. We solve this using **Binary Exponentiation**, an algorithm that calculates powers in $O(\log(\text{exponent}))$ time instead of linear time. It works by exploiting the properties of exponents based on whether the exponent is even or odd:
* If the exponent $b$ is even: $a^b = (a^{b/2})^2$
* If the exponent $b$ is odd: $a^b = a \cdot (a^{(b-1)/2})^2$

This recursive halving process drastically reduces the number of multiplications required to compute the modular inverse.

## 3. Deriving the Slope Formula ($m$)

To add two points $P(x_1, y_1)$ and $Q(x_2, y_2)$, we calculate the slope $m$ of the connecting line, replacing standard division with the modular inverse.

### Case A: Point Addition ($P \neq Q$)
When adding two distinct points, we use the standard algebraic secant line slope, adapted for arithmetic in $\mathbb{F}_p$:
$$m \equiv (y_2 - y_1) \cdot (x_2 - x_1)^{-1} \pmod p$$

### Case B: Point Doubling ($P = Q$)
When adding a point to itself ($P + P$), we cannot use the secant formula. Instead, we compute the tangent line slope using implicit differentiation on the curve equation. In the finite field, this transforms into:
$$m \equiv (3x_1^2 + a) \cdot (2y_1)^{-1} \pmod p$$

## 4. Calculating the Resulting Point $R(x_3, y_3)$

Once we establish the slope $m$, we find the third intersection of the line and the curve. According to Vieta's Formulas applied to the cubic intersection equation, the sum of the $x$-roots equals $m^2$. Adapted for modular arithmetic, the new $x$-coordinate is:
$$x_3 \equiv (m^2 - x_1 - x_2) \pmod p$$

To find the final $y$-coordinate, geometric elliptic curve addition dictates that we reflect the intersection point across the horizontal axis (which means negating the $y$-value). Modulo $p$, this is evaluated as:
$$y_3 \equiv (m(x_1 - x_3) - y_1) \pmod p$$

## 5. Scalar Multiplication: The Double-and-Add Algorithm

In elliptic curve applications, we frequently need to add a point $P$ to itself $k$ times to yield $k \cdot P$. Doing this sequentially ($P + P + P \dots$) is impossible for huge scalars. 

We use the **Double-and-Add** algorithm, which is the geometric equivalent of binary exponentiation. It relies on the binary representation of the scalar $k$:
1. We conceptually express $k$ as a sum of powers of 2 (e.g., $13P = 8P + 4P + 1P$).
2. We continuously **Double** the point dynamically ($P \rightarrow 2P \rightarrow 4P \rightarrow 8P$).
3. We **Add** the currently doubled point to a running accumulator only if the corresponding bit in $k$'s binary representation is a $1$.

This approach elegantly mimics the bit-shifting logic of binary exponentiation, reducing the complexity of scalar multiplication from $O(k)$ to $O(\log k)$. -->

# Elliptic Curve Arithmetic: Reals ($\mathbb{R}$) and Finite Fields ($\mathbb{F}_p$)

## 1. The Curve Equation and Singularity Check

An elliptic curve over the **real numbers ($\mathbb{R}$)** is defined by the Weierstrass equation:
$$y^2 = x^3 + ax + b$$

For the curve to form a valid, non-singular group (meaning it has no cusps or self-intersections), its discriminant must not be zero:
$$\Delta = -4a^3 - 27b^2 \neq 0$$

### Extension to Finite Fields ($\mathbb{F}_p$)
In cryptography, we restrict the curve to a finite field defined by a prime number $p > 3$. The equation becomes a congruence modulo $p$:
$$y^2 \equiv x^3 + ax + b \pmod p$$
Similarly, the curve must remain non-singular in this field, meaning the discriminant must not be congruent to zero modulo $p$:
$$(4a^3 + 27b^2) \not\equiv 0 \pmod p$$

## 2. Modular Arithmetic Fundamentals for $\mathbb{F}_p$

While addition, subtraction, and multiplication in $\mathbb{F}_p$ follow standard arithmetic followed by a modulo operation, **division** does not exist in the same way it does in $\mathbb{R}$. Instead of dividing by a number $k$, we multiply by its **modular multiplicative inverse**, denoted as $k^{-1}$, such that:
$$k \cdot k^{-1} \equiv 1 \pmod p$$

### Fermat's Little Theorem
To find this inverse, we rely on Fermat's Little Theorem, which states that for any integer $k$ not divisible by a prime $p$:
$$k^{p-1} \equiv 1 \pmod p$$
By dividing both sides by $k$ (conceptually factoring out one $k$), we isolate the inverse:
$$k \cdot k^{p-2} \equiv 1 \pmod p \implies k^{-1} \equiv k^{p-2} \pmod p$$

### Binary Exponentiation
Calculating $k^{p-2}$ naively requires $p-2$ multiplications, which is computationally infeasible for the large primes used in cryptography. To solve this, we use **Binary Exponentiation**, an algorithm that computes $a^b \pmod p$ in $O(\log b)$ time. 
It works by observing the binary representation of the exponent $b$. The algorithm recursively (or iteratively) squares the base and halves the exponent. If the current exponent is odd, it multiplies the accumulated result by the current base. This drastically reduces the number of operations required to find the modular inverse.

## 3. Deriving the Slope Formula ($m$)

To add two points $P(x_1, y_1)$ and $Q(x_2, y_2)$ geometrically, we draw a line connecting them. The slope $m$ of this line is calculated differently depending on whether the points are distinct.

### Case A: Point Addition ($P \neq Q$)
**Over $\mathbb{R}$:** We use standard algebra to find the slope of the secant line intersecting both points:
$$m = \frac{y_2 - y_1}{x_2 - x_1}$$

**Over $\mathbb{F}_p$:** The division is replaced by multiplication with the modular inverse of the denominator:
$$m \equiv (y_2 - y_1) \cdot (x_2 - x_1)^{-1} \pmod p$$

### Case B: Point Doubling ($P = Q$)
**Over $\mathbb{R}$:** We cannot use the secant formula since $x_1 = x_2$, resulting in division by zero. Instead, we compute the slope of the **tangent line** at point $P$ using implicit differentiation.
$$2y \cdot \frac{dy}{dx} = 3x^2 + a \implies m = \frac{3x_1^2 + a}{2y_1}$$

**Over $\mathbb{F}_p$:** Applying the same algebraic geometry principles, the tangent slope modulo $p$ becomes:
$$m \equiv (3x_1^2 + a) \cdot (2y_1)^{-1} \pmod p$$

## 4. Calculating the Resulting Point $R(x_3, y_3)$

Once we establish the slope $m$ passing through $P(x_1, y_1)$, the equation of the line is $y = m(x - x_1) + y_1$. To find where this straight line intersects the elliptic curve a third time, we substitute the line equation into the curve equation:
$$(m(x - x_1) + y_1)^2 = x^3 + ax + b$$

Expanding the left side and shifting all terms to one side generates a cubic equation:
$$x^3 - m^2x^2 + \dots = 0$$

According to **Vieta's Formulas**, the sum of the roots of a cubic equation $Ax^3 + Bx^2 + Cx + D = 0$ equals $-B/A$. Our three roots are $x_1, x_2,$ and $x_{int}$.
$$x_1 + x_2 + x_{int} = m^2 \implies x_{int} = m^2 - x_1 - x_2$$

Geometric addition dictates that we **reflect** this intersection across the x-axis ($y \to -y$) to obtain the final point $R(x_3, y_3)$.

**Over $\mathbb{R}$:**
$$x_3 = m^2 - x_1 - x_2$$
$$y_3 = m(x_1 - x_3) - y_1$$

**Over $\mathbb{F}_p$:**
The exact same algebraic structure applies, strictly bounded by the modulo:
$$x_3 \equiv (m^2 - x_1 - x_2) \pmod p$$
$$y_3 \equiv (m(x_1 - x_3) - y_1) \pmod p$$

## 5. Scalar Multiplication ($k \cdot P$)

Scalar multiplication is the process of adding a point $P$ to itself $k$ times ($P + P + \dots + P$). Since doing this sequentially is highly inefficient ($O(k)$ time), we use the **Double-and-Add Algorithm**.

This algorithm is the geometric equivalent of binary exponentiation. It relies on the binary representation of the scalar $k$. For example, if $k = 13$ (which is $1101_2$ in binary), we can express $13P$ as:
$$13P = 8P + 4P + 0P + 1P$$

**Mathematical Justification of the Algorithm:**
1. We evaluate the bits of $k$ sequentially (typically using bitwise right shifts or integer division by 2).
2. **Double:** In every step, regardless of the bit, we double the current base point ($P \to 2P \to 4P \to 8P$).
3. **Add:** If the current bit is $1$ (meaning $k$ is odd), we add the current base point to our accumulated result. If the bit is $0$, we skip the addition.

This approach exploits the associative property of elliptic curve addition, reducing the computational complexity from $O(k)$ to $O(\log k)$, making cryptographic operations fast and secure.

In [2]:
import math

In [ ]:
class EllipticCurveReal:
    def __init__(self, a: float, b:float):
        # Check if the curve is singular
        if(abs(-4*a**3 - 27*b**2) <1e-9):
            raise ValueError("Curve is singular")
        
        self.a = a
        self.b = b

    def is_on_curve(self, P: tuple) -> bool:
        x = P[0]
        y = P[1]

        if(x == None and y == None):
            return True
        left_side = y**2
        right_side = x**3 + self.a * x + self.b

        return math.isclose(left_side, right_side, rel_tol=1e-9, abs_tol=1e-9)

    def add_points(self, p1: tuple, p2: tuple) -> tuple:
        """
        Args:
            p1 (tuple): The first point to add
            p2 (tuple): The second point to add

        Returns:
            tuple: The result of the addition
        """

        if not self.is_on_curve(p1):
            raise ValueError("Point not on curve")
        if not self.is_on_curve(p2):
            raise ValueError("Point not on curve")

        if p1 == (None, None):
            return p2
        if p2 == (None, None):
            return p1

        x1, y1 = p1
        x2, y2 = p2

        if math.isclose(x1, x2, rel_tol=1e-9, abs_tol=1e-9) and math.isclose(y1, -y2, rel_tol=1e-9, abs_tol=1e-9):
            return (None, None)

        if p1 != p2:
            m = (y2 - y1) / (x2 - x1)
        else:
            #Case when p1 == p2 and we use the 
            m = (3 * x1**2 + self.a) / (2 * y1)

        x3 = m**2 - x1 - x2 #By substituting m in the line equation
        y3 = m * (x3 - x1) + y1 #substitute x3 in the line equation to get y3 and obtain the (-) for reflection

        return (x3, -y3)


In [ ]:
class EllipticCurveFp:
    def __init__(self, a: int, b: int, p: int):
        # Check if the curve is singular
        if (4 * a**3 + 27 * b**2) % p == 0:
            raise ValueError("Curve is singular modulo p")
        
        self.p = p
        self.a = a
        self.b = b

    def binpow(self, a: int, b: int) -> int:
        """
        Args:
            a (int): The base
            b (int): The exponent

        Returns:
            int: The result of (a^b) mod p
        """
        if(b == 0):
            return 1
        elif(b%2 == 0):
            tmp = self.binpow(a, b//2) % self.p
            return tmp * tmp % self.p
        else:
            tmp = self.binpow(a, (b-1)//2) % self.p
            return tmp * tmp * a % self.p

    def modular_inverse(self, k: int) -> int:
        """
        Args:
            k (int): The number to find the modular inverse of
            p (int): The modulus

        Returns:
            int: The modular inverse of k modulo p
        """
        if k % self.p == 0:
            raise ValueError("No inverse exists")

        return self.binpow(k, self.p-2) % self.p

    def is_on_curve(self, P: tuple) -> bool:
        if P == (None, None):
            return True
        x = P[0]
        y = P[1]

        left = y*y % self.p
        right = ((x*x % self.p * x % self.p) + (self.a*x % self.p) + self.b) % self.p

        return left == right 

    def add_points(self, p1: tuple, p2: tuple) -> tuple:
        if not self.is_on_curve(p1):
            raise ValueError("Point not on curve")
        if not self.is_on_curve(p2):
            raise ValueError("Point not on curve")

        if p1 == (None, None):
            return p2
        if p2 == (None, None):
            return p1

        x1, y1 = p1
        x2, y2 = p2


        if((x1 == x2) and ((y1+y2)% self.p == 0)):
            return (None, None)

        if(p1 != p2):
            dy = (y2 - y1) % self.p
            dx = (x2 - x1) % self.p
            m = (dy * self.modular_inverse(dx)) % self.p
        else:
            dy = (3 * x1 * x1 + self.a) % self.p
            dx = (2 * y1) % self.p
            m = (dy * self.modular_inverse(dx)) % self.p

        x3 = (m*m - x1 - x2) % self.p
        y3 = ((m*(x1-x3) % self.p) -y1) % self.p

        return (x3, y3)

    def scalar_multiply(self,k:int, P: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError("Point not on curve")

        result = (None, None)
        current_point = P

        while k > 0:
            if (k%2 == 1): #odd
                result = self.add_points(result, current_point)

            current_point = self.add_points(current_point, current_point)

            k = (k//2) #B-shift 

        return result

In [15]:
curve_fp = EllipticCurveFp(a=2, b=2, p=17)